# Plain-Language Notices — LoRA fine-tune (Colab/Kaggle GPU)

Fine-tunes a small instruct model to rewrite bureaucratic benefit notices into plain
language while preserving operative facts. Uses Unsloth + TRL (QLoRA). A free Colab T4
is enough. Settings mirror `train/config.yaml`.

**Flow:** install → get the repo → load the model → train → save the adapter → produce
the `fine_tuned` eval row. The eval row needs **no API key** (readability and the
faithfulness gate are deterministic).

Before running: Runtime → Change runtime type → GPU.

In [ ]:
# 1. Install (Colab/Kaggle)
%pip install -q unsloth "trl>=0.9" "peft>=0.11" "transformers>=4.44" "datasets>=2.20" pydantic

In [ ]:
# 2. Get the repo (code + data)
import os
if not os.path.isdir('repo'):
    !git clone -q https://github.com/adamselzer/plain-language-notices.git repo
print('train pairs:', sum(1 for _ in open('repo/data/pairs.jsonl')))

## Load the base model in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=13,
)

## Format the pairs as chat examples
Each example is (system, user = bureaucratic notice, assistant = plain rewrite), rendered
with the model's chat template. System and user mirror `src/rewrite.py` so training and
inference match.

In [ ]:
import json
from datasets import Dataset

SYSTEM = ('You rewrite government benefit notices into plain language at a sixth-grade '
          'reading level, preserving every operative fact (amounts, dates, the action, '
          'the reason, and appeal/hearing rights with the deadline).')

rows = [json.loads(l) for l in open('repo/data/pairs.jsonl') if l.strip()]
def to_text(r):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':'Rewrite this benefit notice in plain language:\n\n'+r['original']},
            {'role':'assistant','content':r['target']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)
ds = Dataset.from_list([{'text': to_text(r)} for r in rows])
print(ds[0]['text'][:400])

## Train (TRL SFTTrainer)
Argument names are version-tolerant. Current TRL (v1.x, verified against the docs June 2026)
uses `processing_class=` and `SFTConfig(max_length=...)`; older TRL used `tokenizer=` and
`max_seq_length=`. This cell picks whichever the installed version exposes, so it survives
TRL's API churn.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

cfg_kwargs = dict(
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    num_train_epochs=3, learning_rate=2e-4, warmup_steps=5,
    logging_steps=5, seed=13, output_dir='outputs', optim='adamw_8bit',
)
_cfg = inspect.signature(SFTConfig).parameters
cfg_kwargs['max_length' if 'max_length' in _cfg else 'max_seq_length'] = MAX_SEQ
if 'dataset_text_field' in _cfg:
    cfg_kwargs['dataset_text_field'] = 'text'
args = SFTConfig(**cfg_kwargs)

_trn = inspect.signature(SFTTrainer).parameters
tok_kw = {('processing_class' if 'processing_class' in _trn else 'tokenizer'): tokenizer}
trainer = SFTTrainer(model=model, train_dataset=ds, args=args, **tok_kw)
trainer.train()

## Save the adapter
Saved into the cloned repo at `repo/adapters/plain-notices-lora`, where `src/rewrite.py`
and `eval/run_eval.py --with-finetuned` look for it. Download it to drop into your local
clone.

In [ ]:
import os
os.makedirs('repo/adapters', exist_ok=True)
model.save_pretrained('repo/adapters/plain-notices-lora')
tokenizer.save_pretrained('repo/adapters/plain-notices-lora')
print('saved adapter to repo/adapters/plain-notices-lora')
# To keep it: !cd repo && zip -r ../adapter.zip adapters/plain-notices-lora   (then download adapter.zip)

## Quick sanity check

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [{'role':'system','content':SYSTEM},
        {'role':'user','content':'Rewrite this benefit notice in plain language:\n\n'
         'This correspondence constitutes formal notification that your application has been '
         'denied, predicated upon excess income. You may request a hearing within 90 days.'}]
inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
out = model.generate(input_ids=inputs, max_new_tokens=200)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## Produce the `fine_tuned` eval row
Runs the trained model over the 15 holdout notices and scores them with the repo's own
readability and faithfulness gate (both deterministic, **no API key needed**). Prints a
drop-in table row plus the three lines to update in the repo.

In [ ]:
import sys, time, json
if 'repo' not in sys.path:
    sys.path.insert(0, 'repo')
from src.readability import flesch_kincaid_grade
from src.faithfulness import check_faithfulness
from src.schema import NoticePair
from src.config import TARGET_GRADE

FastLanguageModel.for_inference(model)
holdout = [NoticePair(**json.loads(l)) for l in open('repo/data/holdout.jsonl') if l.strip()]

def ft_rewrite(notice):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':'Rewrite this benefit notice in plain language:\n\n'+notice}]
    ids = tokenizer.apply_chat_template(msgs, return_tensors='pt', add_generation_prompt=True).to('cuda')
    t0 = time.time()
    out = model.generate(input_ids=ids, max_new_tokens=400, use_cache=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip(), time.time() - t0

n = len(holdout); hits = faithful = 0; fk_sum = lat = 0.0
for p in holdout:
    text, dt = ft_rewrite(p.original)
    grade = flesch_kincaid_grade(text)
    fk_sum += grade; lat += dt
    hits += grade <= TARGET_GRADE
    faithful += check_faithfulness(p.operative_facts, text).passed

row = f"| fine_tuned | {hits/n:.0%} | {fk_sum/n:.2f} | {faithful/n:.0%} | {lat/n:.3f}s | $0.00000 |"
print('\n=== fine_tuned eval row (paste into eval/report.md after the rag row) ===\n')
print(row)
print('\nThen remove the pending notes in three places:')
print('  - eval/report.md : add fine_tuned to the "Systems evaluated:" line')
print('  - app/app.py     : the "<li>The fine-tuned row ... awaits the Colab training run.</li>" bullet')
print('  - README.md      : the "fine-tuned pending the Colab run" line')